In [1]:
import requests
import json
import pandas as pd
import os
from tqdm import tqdm

D:\Anaconda\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
D:\Anaconda\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
class whoAmI:
    def __init__(self):
        self.getDim='https://ghoapi.azureedge.net/api/Dimension'
        self.getDimValues='https://ghoapi.azureedge.net/api/DIMENSION/COUNTRY/DimensionValues'
        self.getIndValues='https://ghoapi.azureedge.net/api/Indicator'
        self.getIndFilterVal='https://ghoapi.azureedge.net/api/Indicator?$filter=contains(IndicatorName,'
        self.getIndFilterSpecVal='https://ghoapi.azureedge.net/api/Indicator?$filter=IndicatorName eq '
        self.getIndData='https://ghoapi.azureedge.net/api/'
urls = whoAmI()

In [3]:
DATASET_SAVED = "raw_official"
URL_INDICATORS = "urls"

In [4]:
# Khởi tạo thư mục nếu chưa tồn tại
os.makedirs(DATASET_SAVED, exist_ok=True)
os.makedirs(URL_INDICATORS, exist_ok=True)

In [5]:
# Lấy các bộ dữ liệu đã có
existed = set()
for file in os.listdir(DATASET_SAVED):
    indicator = file.split('.')[0]
    existed.add(indicator)

In [6]:
dataset_indicators = set()
for file in os.listdir(URL_INDICATORS):
    with open(URL_INDICATORS + '/' + file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line in existed:
                continue
            dataset_indicators.add(line)
print("Các indicators:", dataset_indicators)

Các indicators: {'SA_0000001425', 'NCD_CCS_CVD_GUIDE', 'M_Est_cig_curr_std', 'NCD_HYP_PREVALENCE_C', 'TOBACCO_MPOWER_OVERVIEW', 'NCD_CCS_BP_SVY', 'BP_06', 'DEVICES00', 'NCD_HYP_CONTROL_A', 'NCD_CCS_rheumreg', 'BP_05', 'NCD_CCS_rheumfollowup', 'NCD_CCS_CVDPlan', 'TOBACCO_MPOWER_M4_SMOKELESSECIGARETTESSCHOOL', 'M_Est_cig_curr', 'NCD_HYP_TREATMENT_A', 'NCD_HYP_DIAGNOSIS_A', 'M_Est_tob_curr_std', 'M_Est_tob_curr', 'M_Est_smk_curr', 'NCD_CCS_stroke', 'DEVICES05', 'NCD_HYP_CONTROL_C', 'NCD_CCS_alteplase', 'NCD_HYP_PREVALENCE_A', 'NCD_HYP_TREATMENT_C', 'NCD_PAC_ADO', 'NCD_CCS_BPmsmt', 'BP_03', 'NCD_HYP_DIAGNOSIS_C', 'BP_04', 'NCD_CCS_rheum', 'NCD_CCS_riskstrat', 'DEVICES19', 'NCD_PAA', 'SA_0000001444', 'NCD_PAC', 'M_Est_smk_curr_std', 'NCD_CCS_BP_TARGET', 'DEVICES01', 'NCDCCS_COUNTRYRESPONSE', 'DEVICES03'}


In [7]:
def request_data(indicator):
    resp = requests.get(urls.getIndData + '/' + indicator)
    data = json.loads(resp.content)["value"]
    fields = ['ParentLocationCode', 'SpatialDim', 'Dim1', 'Value', 'NumericValue', 'Date', 'IndicatorCode']

    records = []
    for row in data:
        _tempo = dict()
        for field in fields:
            _tempo[field] = row[field]
        records.append(_tempo)

    with open(DATASET_SAVED + '/' + indicator + '.json', "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

In [8]:
# Hàm lấy hết dữ liệu
def retrieve_all():
    for indicator in tqdm(dataset_indicators):
        try:
            request_data(indicator)
        except Exception as e:
            print("Error:", e)

In [9]:
retrieve_all()

100%|██████████████████████████████████████████████████████████████████████████████████| 42/42 [03:52<00:00,  5.54s/it]


In [10]:
# Biến chỉ định lưu trữ
CONCAT_GROUP = "raw_concat"
os.makedirs(CONCAT_GROUP, exist_ok=True)

In [11]:
# Tiến hành tổ chức và nhóm lại dữ liệu vào file gốc theo các nhãn
def prepare_data_through_group(group):
    # Group là nhãn dữ liệu, tên thư mục
    target = group.split('.')[0]
    objectives = set()

    with open(URL_INDICATORS + '/' + group, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            objectives.add(line)
    
    array_data = []
    for file in tqdm(os.listdir(DATASET_SAVED)):
        _label = file.split('.')[0]
        if _label in objectives:
            # Đúng nhãn dữ liệu, tiến hành load ra và ghép lại thành một file hoàn chỉnh
            with open(DATASET_SAVED + '/' + file, "rb") as f:
                data = json.load(f)
                
                # Tiến hành nối dữ liệu vào file
            array_data = array_data + data
                
    concated_result = CONCAT_GROUP + '/' + target + '.json'
    
    print("Tổng số điểm dữ liệu là:", len(array_data))
    with open(concated_result, "w", encoding="utf-8") as f:
        json.dump(array_data, f, ensure_ascii=False, indent=2)
    print("Hoàn thành xử lý:", target)

In [23]:
# Chạy xử lí chỉ số BMI
prepare_data_through_group("BMI.txt")

100%|███████████████████████████████████████████████████████████████████████████████| 178/178 [00:01<00:00, 102.34it/s]


Tổng số điểm dữ liệu là: 432287
Hoàn thành xử lý: BMI


In [24]:
# CHạy xử lí chỉ số Diabetes
prepare_data_through_group("diabetes.txt")

100%|███████████████████████████████████████████████████████████████████████████████| 178/178 [00:01<00:00, 159.17it/s]


Tổng số điểm dữ liệu là: 190774
Hoàn thành xử lý: diabetes


In [25]:
# Chạy xử lí tiêu thụ cồn
prepare_data_through_group("alcohol_consumption.txt")

100%|████████████████████████████████████████████████████████████████████████████████| 178/178 [00:05<00:00, 34.19it/s]


Tổng số điểm dữ liệu là: 250720
Hoàn thành xử lý: alcohol_consumption


In [14]:
# Chạy xử lí air_pollution
prepare_data_through_group("air_pollution.txt")

100%|████████████████████████████████████████████████████████████████████████████████| 243/243 [00:03<00:00, 76.07it/s]


Tổng số điểm dữ liệu là: 491076
Hoàn thành xử lý: air_pollution


In [15]:
# Chạy xử lí cholesterol
prepare_data_through_group("cholesterol.txt")

100%|███████████████████████████████████████████████████████████████████████████████| 243/243 [00:00<00:00, 726.03it/s]


Tổng số điểm dữ liệu là: 94536
Hoàn thành xử lý: cholesterol


In [16]:
# Chạy xử lí Glucose
prepare_data_through_group("glucose.txt")

100%|██████████████████████████████████████████████████████████████████████████████| 243/243 [00:00<00:00, 3421.57it/s]


Tổng số điểm dữ liệu là: 21630
Hoàn thành xử lý: glucose


In [12]:
# Chạy xử lí hoạt động thể chất
prepare_data_through_group("physical_activities.txt")

100%|██████████████████████████████████████████████████████████████████████████████| 285/285 [00:00<00:00, 2079.91it/s]


Tổng số điểm dữ liệu là: 35523
Hoàn thành xử lý: physical_activities


In [13]:
# Chạy xử lí tiếp cận cơ sở y tế
prepare_data_through_group("infrastructure.txt")

100%|███████████████████████████████████████████████████████████████████████████████| 285/285 [00:00<00:00, 398.28it/s]


Tổng số điểm dữ liệu là: 170916
Hoàn thành xử lý: infrastructure


In [14]:
# Chạy xử lí thuốc lá
prepare_data_through_group("tobacco.txt")

100%|███████████████████████████████████████████████████████████████████████████████| 285/285 [00:00<00:00, 895.54it/s]


Tổng số điểm dữ liệu là: 48540
Hoàn thành xử lý: tobacco


In [15]:
# Chạy xử lí bệnh tim mạch
prepare_data_through_group("cardiovascular_diseases.txt")

100%|███████████████████████████████████████████████████████████████████████████████| 285/285 [00:01<00:00, 200.20it/s]


Tổng số điểm dữ liệu là: 236876
Hoàn thành xử lý: cardiovascular_diseases
